# Tăng độ chính xác V5+ không cần train lại — chạy trên GPU Colab

Notebook chạy `scripts/eval/no_retrain_experiments.py` trên tập val (1.466 ảnh), so 7 cách:

| Cách | Ý tưởng |
|---|---|
| `base960` | Suy luận chuẩn imgsz 960 (mốc so sánh) |
| `base960+per_class_conf` | Ngưỡng conf riêng từng lớp (tính hậu kỳ, không chạy thêm) |
| `sz1152`, `sz1280` | Tăng độ phân giải khi suy luận |
| `tta960` | Test-Time Augmentation (lật, đổi tỉ lệ) |
| `clahe960` | Tăng tương phản cho vùng tối / bóng râm |
| `slice960` | Cắt ảnh thành ô (kiểu SAHI) để bắt xe nhỏ ở xa |
| `wbf_v5p_v6` | Gộp dự đoán V5+ và V6 bằng Weighted Boxes Fusion |

**Chống crash RAM:** mỗi cách chạy trong một tiến trình Python riêng (xong là giải phóng
hết bộ nhớ), và dự đoán được lưu ngay lên Drive. Nếu Colab bị ngắt, chạy lại notebook:
cách nào đã xong sẽ được bỏ qua.

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU**.

## 1. Kiểm tra GPU và nối Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -h | head -2
from google.colab import drive
drive.mount('/content/drive')

## 2. Cấu hình và kiểm tra file

In [ ]:
import os, subprocess, sys
from pathlib import Path

PROJECT_DIR = "/content/drive/MyDrive/Đồ án TGMT nhóm 2/VN_Traffic_AI"
DATA_DIR    = "/content/merged_v6"          # giải nén ra ổ Colab cho nhanh, không lên Drive
OUT_DIR     = f"{PROJECT_DIR}/results/no_retrain_gpu"   # lưu lên Drive để không mất khi crash
DEVICE      = "0"
METHODS     = ["base960", "sz1152", "sz1280", "clahe960", "wbf_v5p_v6", "tta960", "slice960"]
MAX_IMAGES  = None     # đặt 200 để chạy thử nhanh trước

P = Path(PROJECT_DIR)
need = ["scripts/eval/no_retrain_experiments.py", "src/evaluation/det_analysis.py",
        "runs/detect/train_v8s_v5plus_960/weights/best.pt",
        "runs/detect/train_v8s_v6_pseudo/weights/best.pt"]
ok = True
for f in need:
    e = (P / f).exists()
    ok &= e
    print("OK   " if e else "THIẾU", f)
assert ok, "Thiếu file: chép từ thư mục SuaDoi_VN_Traffic_AI lên đúng chỗ trên Drive rồi chạy lại ô này."

## 3. Cài Ultralytics

In [ ]:
!pip -q install ultralytics
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime > Change runtime type > T4 GPU"

## 4. Lấy dữ liệu val (chỉ giải nén phần val)

Chỉ tải nếu chưa có. Giải nén riêng `images/val` và `labels/val` để tiết kiệm ổ và RAM,
rồi xoá file nén.

In [ ]:
val_img = Path(DATA_DIR) / "images/val"
if not val_img.exists() or len(os.listdir(val_img)) < 1466:
    tar = "/content/dataset_v6_merged.tar.gz"
    if not Path(tar).exists():
        !wget -q -O {tar} https://github.com/xuanduc24905-beep/VN_Traffic_AI/releases/download/v6.0-pseudo/dataset_v6_merged.tar.gz
    !tar -xzf {tar} -C /content merged_v6/images/val merged_v6/labels/val
    !rm -f {tar}
n_img = len(os.listdir(val_img))
n_lbl = len(os.listdir(Path(DATA_DIR) / "labels/val"))
print(f"{n_img} ảnh | {n_lbl} nhãn")
assert n_img == n_lbl and n_img > 0
!df -h /content | tail -1

## 5. Chạy từng cách — mỗi cách một tiến trình riêng

Cách nào đã có kết quả trong `OUT_DIR/cache/` sẽ được bỏ qua. Nếu muốn chạy lại một cách,
xoá file `.pkl` tương ứng trong thư mục đó.

In [ ]:
import time
SCRIPT = str(P / "scripts/eval/no_retrain_experiments.py")
base = [sys.executable, SCRIPT, "--data", DATA_DIR, "--out", OUT_DIR, "--device", DEVICE]
if MAX_IMAGES:
    base += ["--max-images", str(MAX_IMAGES)]

for m in METHODS:
    if (Path(OUT_DIR) / "cache" / f"{m}.pkl").exists():
        print(f"[{m}] đã có kết quả, bỏ qua")
        continue
    t = time.time()
    r = subprocess.run(base + ["--methods", m], capture_output=True, text=True,
                       env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    lines = [l for l in (r.stdout + r.stderr).splitlines()
             if l.startswith("[") or "Error" in l or "Traceback" in l]
    print("\n".join(lines[-4:]), f"| {time.time() - t:.0f}s")
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-3000:])
        raise SystemExit(f"Lỗi ở {m}")
    !free -h | sed -n 2p

## 6. Tổng hợp kết quả

In [ ]:
r = subprocess.run([sys.executable, SCRIPT, "--data", DATA_DIR, "--out", OUT_DIR, "--report-only"]
                   + (["--max-images", str(MAX_IMAGES)] if MAX_IMAGES else []),
                   capture_output=True, text=True, env={**os.environ, "PYTHONIOENCODING": "utf-8"})
print(r.stdout[-600:] if r.returncode == 0 else r.stderr[-3000:])

import pandas as pd
df = pd.read_csv(Path(OUT_DIR) / "summary.csv")
b = df[df.method == "base960"].iloc[0]
for c in ["mAP50", "mAP50-95", "F1", "AP50_moto_small"]:
    df["Δ" + c] = (df[c] - b[c]).round(4)
show = ["method", "s/img", "mAP50", "mAP50-95", "AP50_motorcycle", "AP50_bus", "AP50_truck",
        "AP50_moto_small", "conf", "F1", "ΔmAP50", "ΔmAP50_CI95", "P(Δ>0)",
        "ΔAP50_moto_small", "ΔF1"]
pd.set_option("display.width", 250)
df[show]

In [ ]:
import matplotlib.pyplot as plt
d = df.set_index("method")
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
d["ΔmAP50"].plot.barh(ax=ax[0], color=["tab:green" if v > 0 else "tab:red" for v in d["ΔmAP50"]])
ax[0].set_title("Thay đổi mAP50 so với base960")
d["ΔAP50_moto_small"].plot.barh(ax=ax[1], color=["tab:green" if v > 0 else "tab:red" for v in d["ΔAP50_moto_small"]])
ax[1].set_title("Thay đổi AP50 xe máy nhỏ (< 32x32 px)")
ax[2].scatter(d["s/img"] * 1000, d["mAP50"])
for n, row in d.iterrows():
    ax[2].annotate(n, (row["s/img"] * 1000, row["mAP50"]), fontsize=8)
ax[2].set_xlabel("ms / ảnh (GPU)"); ax[2].set_ylabel("mAP50"); ax[2].set_title("Độ chính xác và tốc độ")
for a in ax:
    a.grid(alpha=.3)
plt.tight_layout()
plt.savefig(Path(OUT_DIR) / "no_retrain_summary.png", dpi=150)
plt.show()

## Đọc kết quả

- **mAP50 / mAP50-95** tính trên toàn bộ val, kiểu COCO 101 điểm, mỗi box một lớp
  (giống lúc dùng thật). Số này thấp hơn `model.val` của Ultralytics khoảng 0,02–0,03
  vì `model.val` dùng NMS đa nhãn (`multi_label=True`) — đó là khác biệt cách đo,
  không phải model kém đi.
- **AP50_moto_small**: AP của riêng xe máy nhỏ (< 32×32 px), không phụ thuộc ngưỡng conf.
  Dùng cột này để so khả năng bắt xe máy ở xa giữa các cách. Cột `R_moto_small` thì
  phụ thuộc ngưỡng mỗi cách tự chọn nên không so trực tiếp được.
- **ΔmAP50_CI95, P(Δ>0)**: khoảng tin cậy 95% của mức chênh mAP50 so với base960
  (bootstrap 300 lần theo ảnh) và tỉ lệ lần bootstrap mà cách đó tốt hơn base.
  Khoảng không chứa 0 thì mức chênh đáng tin.
- **conf, P, R, F1**: ngưỡng chọn trên một nửa val (tune), đo trên nửa
  còn lại (eval), để không chọn và đo trên cùng ảnh.
- **s/img** đo trên GPU Colab, gồm cả đọc ảnh — dùng cho cột tốc độ trong báo cáo.
- Ảnh Cần Thơ chỉ 416×416 nên `slice960` không cắt ô với các ảnh này; lợi ích của
  cắt ô chỉ đến từ ảnh SOICT 1280×720.